# Basketball-v0.1 Model Comparison

This notebook consumes immutable evaluation bundles below
`OUTPUT_ROOT/evaluation/basketball_large_dataset/`. It does not load model
frameworks, start producer processes, run inference, train, or require a GPU.
Missing models remain unavailable until their owning project finishes or
recovers a new-protocol full run and publishes its bundle.

In [1]:
from __future__ import annotations

from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display

from detection_evaluation import (
	discover_bundles,
	write_bundle_comparisons,
)
from detection_evaluation.config import DATA_ROOT, OUTPUT_ROOT, WORKSPACE_ROOT

## Comparison settings

Leave a model out of `RUN_SELECTIONS` to select its newest compatible full
bundle by protocol creation time and run name. Set a value to a run name or
absolute bundle directory to require that exact published bundle. Explicit
missing, smoke, truncated, or invalid selections raise instead of falling
back to another run.

In [2]:
RUN_SELECTIONS: dict[str, str | Path] = {
	# "yolo11n": "yolo11n_20260919T120000",
	# "yolox_tiny": "yolox_tiny_20260919T130000",
	# "yolox_nano": "yolox_nano_20260919T140000",
	# "rfdetr_small": "rfdetr_small_20260919T150000",
}
SOURCE_DATASET = DATA_ROOT / "composed" / "basketball-v0.1" / "coco"
REPORT_DIR = OUTPUT_ROOT / "comparisons" / "basketball_large_dataset"
ANNOTATIONS = {
	split: SOURCE_DATASET / "annotations" / f"instances_{split}.json"
	for split in ("val", "test")
}

## Discover and validate bundles

Each candidate is read once through the Stage 3 reader. The consumer verifies
the immutable generation, hashes, model and split mapping, selected checkpoint,
resolution, parameter count, training summary, and canonical source identity
before comparison. Smoke and truncated candidates are excluded automatically.

In [3]:
selections = discover_bundles(OUTPUT_ROOT, selections=RUN_SELECTIONS)
selection_rows = []
for model, selection in selections.items():
	details = selection.report_details()
	summary = details.get("training_summary", {})
	selection_rows.append(
		{
			"model": model,
			"bundle_dir": (
				Path(details["bundle_dir"]).relative_to(WORKSPACE_ROOT)
				if details.get("bundle_dir")
				else None
			),
			"parameters": details.get("parameter_count"),
			"completed_epochs": summary.get("completed_epochs"),
			"training_hours": summary.get("total_hours"),
		}
	)
with pd.option_context("display.max_colwidth", None):
	display(pd.DataFrame(selection_rows))

,model,bundle_dir,parameters,completed_epochs,training_hours
0,yolo11n,outputs/evaluation/basketball_large_dataset/yolo11n/yolo11n_20261105T160703,2582347,100,1.019770
1,yolox_tiny,outputs/evaluation/basketball_large_dataset/yolox_tiny/yolox_tiny_20261105T170915,5032866,100,2.892357
2,yolox_nano,outputs/evaluation/basketball_large_dataset/yolox_nano/yolox_nano_20261105T200341,896754,100,2.860266
3,rfdetr_small,outputs/evaluation/basketball_large_dataset/rfdetr_small/rfdetr_small_20261106T124419,32012132,38,1.460188


## Recompute validation and held-out metrics

Both reports use the shared COCO evaluator. AP uses a score floor of 0.001;
precision, recall, F1, and negative-image error measures use confidence 0.25
and IoU 0.50. Inference timings are compared only when their benchmark
protocol, host, accelerator, image sequence, and settings are compatible.

In [4]:
reports = write_bundle_comparisons(selections, ANNOTATIONS, REPORT_DIR)
for split, report in reports.items():
	display_report = report.copy()
	for column in ("run_dir", "checkpoint"):
		if column in display_report:
			display_report[column] = display_report[column].map(
				lambda value: Path(value).relative_to(WORKSPACE_ROOT)
				if pd.notna(value)
				else value
			)
	display(Markdown(f"## {split.title()} metrics"))
	with pd.option_context("display.max_colwidth", None):
		display(display_report)

## Val metrics

,model,split,status,run_dir,checkpoint,resolution,parameters,epochs_completed,ap50,ap50_95,...,benchmark_samples,latency_ms_median,latency_ms_mean,batch_one_images_per_second,training_status,training_timing_available,training_total_seconds,training_total_hours,training_completed_epochs,training_amortized_seconds_per_completed_epoch
0,yolo11n,val,available,outputs/runs/basketball/yolo11n_20261105T160703,outputs/runs/basketball/yolo11n_20261105T160703/weights/best.pt,640,NaN,NaN,0.952472,0.723906,...,32,3.273956,3.268692,305.440878,completed,True,3671.170379,1.019770,100,36.711704
1,yolox_tiny,val,available,outputs/runs/basketball/yolox_tiny_20261105T170915,outputs/runs/basketball/yolox_tiny_20261105T170915/weights/best_ckpt.pth,640,NaN,NaN,0.968192,0.719435,...,32,5.484309,5.833306,182.338379,completed,True,10412.484817,2.892357,100,104.124848
2,yolox_nano,val,available,outputs/runs/basketball/yolox_nano_20261105T200341,outputs/runs/basketball/yolox_nano_20261105T200341/weights/best_ckpt.pth,640,NaN,NaN,0.958437,0.695038,...,32,6.235772,6.239634,160.365070,completed,True,10296.956444,2.860266,100,102.969564
3,rfdetr_small,val,available,outputs/runs/basketball/rfdetr_small_20261106T124419,outputs/runs/basketball/rfdetr_small_20261106T124419/checkpoint_best_total.pth,640,32012132.0,38.0,0.968490,0.757875,...,32,8.663272,8.660866,115.429827,completed,True,5256.678505,1.460188,38,138.333645


## Test metrics

,model,split,status,run_dir,checkpoint,resolution,parameters,epochs_completed,ap50,ap50_95,...,benchmark_samples,latency_ms_median,latency_ms_mean,batch_one_images_per_second,training_status,training_timing_available,training_total_seconds,training_total_hours,training_completed_epochs,training_amortized_seconds_per_completed_epoch
0,yolo11n,test,available,outputs/runs/basketball/yolo11n_20261105T160703,outputs/runs/basketball/yolo11n_20261105T160703/weights/best.pt,640,NaN,NaN,0.958327,0.719457,...,32,3.279806,3.277751,304.896082,completed,True,3671.170379,1.019770,100,36.711704
1,yolox_tiny,test,available,outputs/runs/basketball/yolox_tiny_20261105T170915,outputs/runs/basketball/yolox_tiny_20261105T170915/weights/best_ckpt.pth,640,NaN,NaN,0.970393,0.719501,...,32,5.587199,5.581687,178.980574,completed,True,10412.484817,2.892357,100,104.124848
2,yolox_nano,test,available,outputs/runs/basketball/yolox_nano_20261105T200341,outputs/runs/basketball/yolox_nano_20261105T200341/weights/best_ckpt.pth,640,NaN,NaN,0.964523,0.699667,...,32,7.461297,7.360639,134.024948,completed,True,10296.956444,2.860266,100,102.969564
3,rfdetr_small,test,available,outputs/runs/basketball/rfdetr_small_20261106T124419,outputs/runs/basketball/rfdetr_small_20261106T124419/checkpoint_best_total.pth,640,32012132.0,38.0,0.965659,0.759730,...,32,8.515843,8.524852,117.428187,completed,True,5256.678505,1.460188,38,138.333645


## Saved reports and interpretation

The report directory contains the selected bundle provenance plus metric,
inference, and training CSV/JSON/Markdown views for both splits. Training
totals are derived only from finalized monotonic attempts; unavailable timing
remains unavailable rather than being reconstructed.

In [5]:
for name in ("selection", "metrics_test", "inference_test", "training_test"):
	report_path = REPORT_DIR / f"{name}.md"
	if report_path.is_file():
		report_text = report_path.read_text().replace(f"{WORKSPACE_ROOT}/", "")
		display(Markdown(report_text))

# Basketball bundle selection

| Model | Status | Bundle | Generation | Checkpoint SHA256 | Recovery |
| --- | --- | --- | --- | --- | --- |
| yolo11n | available | outputs/evaluation/basketball_large_dataset/yolo11n/yolo11n_20261105T160703 | 651bbc93cae041a1be2bbc01534f2c78 | 982bb81728373331e16d967c29f4b38527111a0110569ed87fce3918697deb23 | - |
| yolox_tiny | available | outputs/evaluation/basketball_large_dataset/yolox_tiny/yolox_tiny_20261105T170915 | f8a7fc29195d460d84bc02019d8f2d48 | eedefa6a70ad9fae1af66154a8326851ecb2f1abe6dbea32381740725e036edc | - |
| yolox_nano | available | outputs/evaluation/basketball_large_dataset/yolox_nano/yolox_nano_20261105T200341 | ac738fa562134c83b7f280b620bb756f | d9b9d32ee716c90480bd1bb1a40304c6ee0fed064d6d996d2876ea5a470e99d3 | - |
| rfdetr_small | available | outputs/evaluation/basketball_large_dataset/rfdetr_small/rfdetr_small_20261106T124419 | 3a8357af76b7483f9cf8993de462e715 | 85fc2abfa9fd2aa0964c7b355900711aae8f0c6185776f63bfefc597bd287c64 | - |


# Basketball metrics: test

| model | status | ap50 | ap50 95 | ar100 | precision | recall | f1 | false positives per negative image | negative image false positive fraction |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| yolo11n | available | 0.9583270786887065 | 0.7194565512499935 | 0.7614942528735633 | 0.9247191011235955 | 0.9459770114942528 | 0.9352272727272726 | 0.07746478873239436 | 0.07042253521126761 |
| yolox_tiny | available | 0.9703927075021758 | 0.7195012510853899 | 0.7603448275862068 | 0.9543899657924744 | 0.9620689655172414 | 0.9582140812821981 | 0.04929577464788732 | 0.04929577464788732 |
| yolox_nano | available | 0.9645233919245978 | 0.6996668263594422 | 0.7462068965517241 | 0.9466515323496028 | 0.9586206896551724 | 0.9525985151342091 | 0.06338028169014084 | 0.056338028169014086 |
| rfdetr_small | available | 0.9656594214190103 | 0.7597298445029858 | 0.813103448275862 | 0.9596774193548387 | 0.957471264367816 | 0.9585730724971232 | 0.04929577464788732 | 0.04225352112676056 |


# Basketball inference: test

| model | status | resolution | parameters | benchmark samples | latency ms median | latency ms mean | batch one images per second |
| --- | --- | --- | --- | --- | --- | --- | --- |
| yolo11n | available | 640 | nan | 32 | 3.2798059983178973 | 3.2777513115433976 | 304.89608242465147 |
| yolox_tiny | available | 640 | nan | 32 | 5.587198538705707 | 5.581687204539776 | 178.98057372982728 |
| yolox_nano | available | 640 | nan | 32 | 7.46129744220525 | 7.360638948739506 | 134.0249477716092 |
| rfdetr_small | available | 640 | 32012132.0 | 32 | 8.515842957422137 | 8.524852026312146 | 117.42818708609838 |


# Basketball training: test

| model | status | training status | training timing available | training total seconds | training total hours | training completed epochs | training amortized seconds per completed epoch |
| --- | --- | --- | --- | --- | --- | --- | --- |
| yolo11n | available | completed | True | 3671.1703793420456 | 1.019769549817235 | 100 | 36.711703793420455 |
| yolox_tiny | available | completed | True | 10412.484816684155 | 2.8923568935233765 | 100 | 104.12484816684155 |
| yolox_nano | available | completed | True | 10296.956444045994 | 2.860265678901665 | 100 | 102.96956444045995 |
| rfdetr_small | available | completed | True | 5256.678504718933 | 1.460188473533037 | 38 | 138.33364486102457 |
